In [ ]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler
import xmlrpc.client
import datetime
import os
import csv
import pickle


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

server = SimpleXMLRPCServer(("127.0.0.1", 8009),
                            requestHandler=RequestHandler)


def log_event(event_type: str, details: str = ""):
    try:
        stats = xmlrpc.client.ServerProxy("http://127.0.0.1:8018", allow_none=True)
        stats.register_event(event_type, details)
    except Exception:
        pass


# Тест
def ping():
    log_event("ping")
    return True
server.register_function(ping, 'ping')


def now():
    log_event("now")
    return datetime.datetime.now().isoformat()
server.register_function(now, 'now')


def show_type(arg):
    log_event("type", str(type(arg)))
    return str(arg), str(type(arg)), arg
server.register_function(show_type, 'type')


def test_sum(a, b):
    log_event("sum", f"{a}+{b}")
    return a + b
server.register_function(test_sum, 'sum')


def test_pow(a, b):
    log_event("pow", f"{a}^{b}")
    return a ** b
server.register_function(test_pow, 'pow')


BLACK_LIST_FILE = "bad_boys2.csv"


def _levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                prev[j] + 1,
                cur[j - 1] + 1,
                prev[j - 1] + (ca != cb)
            ))
        prev = cur
    return prev[-1]


def _load_black_list():
    rows = []
    if not os.path.exists(BLACK_LIST_FILE):
        return rows
    with open(BLACK_LIST_FILE, encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append({
                "Surname":  (r.get("Surname")  or "").strip(),
                "Name":     (r.get("Name")     or "").strip(),
                "Patronym": (r.get("Patronym") or "").strip(),
                "Birth":    (r.get("Birth")    or "").strip(),
                "City":     (r.get("City")     or "").strip(),
                "Position": (r.get("Position") or "").strip(),
            })
    return rows


def black_list_check(sname):
    log_event("black_list_check", f"Surname={sname}")

    if not isinstance(sname, str):
        return ["no"]

    q = sname.strip().lower()
    rows = _load_black_list()

    exact, similar = [], []
    for r in rows:
        sn = r["Surname"].lower()
        if sn == q:
            exact.append(r)
        elif _levenshtein(sn, q) <= 1:
            similar.append(r)

    if exact:
        return [f"{sname}: yes", exact]
    elif similar:
        return [f"{sname}: similar", similar]
    else:
        return [f"{sname}: no"]

server.register_function(black_list_check, 'black_list_check')


def black_list_check_fio(surname, name, patronymic, birth_date):
    log_event("black_list_check_fio",
              f"{surname} {name} {patronymic} {birth_date}")

    if not all(isinstance(x, str)
               for x in (surname, name, patronymic, birth_date)):
        return ["no"]

    s_in = surname.strip().lower()
    n_in = name.strip().lower()
    p_in = patronymic.strip().lower()
    b_in = birth_date.strip()

    rows = _load_black_list()

    exact, similar = [], []
    for r in rows:
        sn = r["Surname"].lower()
        nm = r["Name"].lower()
        pt = r["Patronym"].lower()
        bd = r["Birth"].strip()

        full_match = (sn == s_in and nm == n_in and pt == p_in)
        same_birth = (bd == b_in)

        if full_match and same_birth:
            exact.append(r)
        elif full_match or (
                same_birth and
                _levenshtein(sn, s_in) <= 1 and
                _levenshtein(nm, n_in) <= 1 and
                _levenshtein(pt, p_in) <= 1):
            similar.append(r)

    fio = f"{surname} {name} {patronymic}, {birth_date}"
    if exact:
        return [f"{fio}: yes", exact]
    elif similar:
        return [f"{fio}: similar", similar]
    else:
        return [f"{fio}: no"]

server.register_function(black_list_check_fio, 'black_list_check_fio')


# Бинарная передача данных
def send_back_binary(bin_data):
    log_event("send_back_binary")
    data = bin_data.data
    return xmlrpc.client.Binary(data)
server.register_function(send_back_binary, 'send_back_binary')


def send_back_inversion(bin_data):
    log_event("color_inversion")
    img_arr = pickle.loads(bin_data.data)

    height = img_arr.shape[0]
    width = img_arr.shape[1]
    channels = img_arr.shape[2] if img_arr.ndim == 3 else 1

    for i in range(height):
        for j in range(width):
            if channels == 1:
                img_arr[i][j][0] = 255 - img_arr[i][j][0]
            else:
                for c in range(channels):
                    img_arr[i][j][c] = 255 - img_arr[i][j][c]

    return xmlrpc.client.Binary(pickle.dumps(img_arr))
server.register_function(send_back_inversion, 'color_inversion')


def binarization_percent(bin_data, threshold):
    log_event("binarization_percent", f"threshold={threshold}")

    img_arr = pickle.loads(bin_data.data)

    if threshold < 1 or threshold > 255:
        raise ValueError("Порог должен быть в диапазоне 1-255")

    height = img_arr.shape[0]
    width = img_arr.shape[1]
    channels = img_arr.shape[2] if img_arr.ndim == 3 else 1

    total = height * width
    above = 0

    for i in range(height):
        for j in range(width):
            if channels == 1:
                val = int(img_arr[i][j][0])
            else:
                s = sum(int(img_arr[i][j][c]) for c in range(channels))
                val = s // channels

            if val >= threshold:
                above += 1
                result = 255
            else:
                result = 0

            if channels == 1:
                img_arr[i][j][0] = result
            else:
                for c in range(channels):
                    img_arr[i][j][c] = result

    percent = round(above / total * 100.0, 2)
    return xmlrpc.client.Binary(pickle.dumps(img_arr)), percent

server.register_function(binarization_percent, 'binarization_percent')


def send_back_flip_vertical(bin_data):
    log_event("flip_vertical")
    img_arr = pickle.loads(bin_data.data)

    height = img_arr.shape[0]
    width = img_arr.shape[1]
    channels = img_arr.shape[2] if img_arr.ndim == 3 else 1

    result = pickle.loads(pickle.dumps(img_arr))

    for i in range(height):
        for j in range(width):
            src_j = width - 1 - j
            if channels == 1:
                result[i][j][0] = img_arr[i][src_j][0]
            else:
                for c in range(channels):
                    result[i][j][c] = img_arr[i][src_j][c]

    return xmlrpc.client.Binary(pickle.dumps(result))
server.register_function(send_back_flip_vertical, 'flip_vertical')



print("Listening on port 8009...")
server.serve_forever()

Listening on port 8009...


127.0.0.1 - - [19/Oct/2026 15:22:12] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:12] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:12] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:12] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:12] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:12] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:12] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:12] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:12] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:12] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:35] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:35] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:35] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:35] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 15:22:35] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/202